In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('titanic.csv'), Path('LAB1') / 'titanic.csv', Path('LAB1') / 'Bai2_Trang15-17' / 'titanic.csv', Path('../titanic.csv'), Path('/content/titanic.csv'), Path('/mnt/data/titanic.csv')]
data_path = next((p for p in candidates if p.is_file()), None)
if data_path is None:
    try:
        from google.colab import files
    except ImportError:
        files = None
    if files is not None:
        print('Chưa tìm thấy titanic.csv. Hãy chọn tệp CSV để tải lên Colab.')
        uploaded = files.upload()
        uploaded_path = next((Path(name) for name in uploaded if Path(name).name.lower() == 'titanic.csv'), None)
        if uploaded_path is not None and uploaded_path.is_file():
            data_path = uploaded_path
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy titanic.csv. Thư mục chạy hiện tại: {Path.cwd()}. Hãy tải CSV lên Colab hoặc đặt CSV trong thư mục này rồi chạy lại ô.')
df = pd.read_csv(data_path)
age_bins = [-float('inf'), 11, 18, 22, 27, 33, 40, 66, float('inf')]
age_labels = ['1: ≤11', '2: 11–18', '3: 18–22', '4: 22–27', '5: 27–33', '6: 33–40', '7: 40–66', '8: >66']
plot_data = df.dropna(subset=['age', 'sex', 'survived']).copy()
plot_data['nhom_tuoi'] = pd.cut(plot_data['age'], bins=age_bins, labels=age_labels, include_lowest=True)
summary = (plot_data.groupby(['nhom_tuoi', 'sex'], observed=True)['survived']
           .agg(ty_le_song='mean', so_nguoi='size').reset_index())
rates = summary.pivot(index='nhom_tuoi', columns='sex', values='ty_le_song').reindex(age_labels)
print('Tỷ lệ sống sót và số người trong từng nhóm:')
display(summary)
ax = rates.plot(kind='bar', figsize=(11, 6), ylim=(0, 1), color={'female': '#d95f8d', 'male': '#4c78a8'})
ax.set_title('Tỷ lệ sống sót theo nhóm tuổi và giới tính')
ax.set_xlabel('Nhóm tuổi')
ax.set_ylabel('Tỷ lệ sống sót')
ax.legend(title='Giới tính')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print('Nhận xét dựa trên dữ liệu:')
print('Tỷ lệ sống sót chung theo giới tính:')
survival_by_sex = (plot_data.groupby('sex')['survived'].mean().sort_values(ascending=False)
                   .round(3).rename_axis('sex').reset_index(name='survival_rate'))
display(survival_by_sex)
for sex, group in summary.groupby('sex'):
    valid = group.dropna(subset=['ty_le_song'])
    if not valid.empty:
        best = valid.loc[valid['ty_le_song'].idxmax()]
        print(f"{sex}: nhóm có tỷ lệ sống cao nhất là {best['nhom_tuoi']} ({best['ty_le_song']:.1%}, n={int(best['so_nguoi'])}).")
print('Nhận xét: đối chiếu các cột theo giới tính và nhóm tuổi; nhóm có ít hành khách cần được diễn giải thận trọng.')